# Parte 1 - Baseline RAG funcional

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña.**

En esta parte se implementa un pipeline RAG baseline sobre un corpus real de documentos.

El flujo general del sistema es:

**Ingesta → Fragmentación → Embeddings → Indexación → Recuperación → Generación**

**Implementa el pipeline sobre tu corpus, a partir del Lab-02 del día miércoles:**

Primero se cargan las librerías y se ubica la raíz del proyecto. Después se fijan las variables de entorno y **solo entonces** se importa `RagPipeline`: `rag_pipeline.py` lee `QDRANT_URL`, `OLLAMA_URL` y `EMBEDDING_BACKEND` una sola vez, al importarse. Si se importa antes, toma los valores del `.env` (en una versión anterior de este notebook eso hizo que la generación usara `localhost:11434` y cayera en *modo inspección* sin ningún error).

In [1]:
import socket
import sys
import os
import re
import json
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

In [2]:
# Buscar la raíz del proyecto (la carpeta con pyproject.toml y rag_pipeline.py)
raiz = Path.cwd()
while raiz != raiz.parent:
    if (raiz / "pyproject.toml").exists() and (raiz / "rag_pipeline.py").exists():
        break
    raiz = raiz.parent

ROOT = raiz
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("Raíz del proyecto:", ROOT)
print("rag_pipeline.py:", (ROOT / "rag_pipeline.py").exists())
print("pyproject.toml:", (ROOT / "pyproject.toml").exists())

Raíz del proyecto: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña
rag_pipeline.py: True
pyproject.toml: True


In [3]:
## Configuración del baseline (ANTES de importar rag_pipeline)

os.environ["EMBEDDING_BACKEND"] = "h200"                  # bge-m3 en la H200 (GlobalProtect)
os.environ["QDRANT_URL"] = "http://localhost:6333"        # el contenedor, no :memory:
os.environ["QDRANT_COLLECTION"] = "mmia6013_rag_baseline"
os.environ["CORPUS_DIR"] = str(ROOT / "corpus")

# Generador ÚNICO para todo el taller: Ollama de la H200, fila open_weight_pequeno (qwen3:32b).
# Se vacían las claves para que generate() no tome la ruta de OpenAI/Anthropic por lo que
# haya en .env (load_dotenv no sobrescribe variables que ya existen).
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OLLAMA_URL"] = "http://172.28.230.10:11434"

for var in ("EMBEDDING_BACKEND", "QDRANT_URL", "QDRANT_COLLECTION", "CORPUS_DIR", "OLLAMA_URL"):
    print(f"{var}: {os.environ[var]}")

EMBEDDING_BACKEND: h200
QDRANT_URL: http://localhost:6333
QDRANT_COLLECTION: mmia6013_rag_baseline
CORPUS_DIR: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/corpus
OLLAMA_URL: http://172.28.230.10:11434


Con las variables fijadas, se importa `rag_pipeline` y se confirma qué leyó.

In [4]:
import rag_pipeline
from rag_pipeline import RagPipeline, EMBEDDING_MODEL, ABSTENCION

print("QDRANT_URL leído por rag_pipeline:", rag_pipeline.QDRANT_URL)
print("OLLAMA_URL leído por rag_pipeline:", rag_pipeline.OLLAMA_URL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("Frase de abstención:", ABSTENCION)

assert rag_pipeline.QDRANT_URL != ":memory:", "La Parte 1 debe indexar en el contenedor"
assert "172.28.230.10" in rag_pipeline.OLLAMA_URL, "OLLAMA_URL no apunta a la H200"

QDRANT_URL leído por rag_pipeline: http://localhost:6333
OLLAMA_URL leído por rag_pipeline: http://172.28.230.10:11434
Modelo de embeddings: BAAI/bge-m3
Frase de abstención: El corpus no contiene información suficiente.


Se requiere que la indexación se realice en un contenedor de Qdrant y no utilizando `:memory:`.

Por ello, antes de crear el pipeline se comprueba que el servicio de Qdrant esté disponible en `localhost:6333`.

In [5]:
cliente_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"]
)

print("Colecciones existentes:")
print(cliente_qdrant.get_collections())

Colecciones existentes:
collections=[CollectionDescription(name='mmia6013_rag_baseline')]


In [6]:
print("Host:", socket.gethostname())


Host: PC-Miguel.local


Una vez configuradas las variables de entorno y comprobada la conexión con Qdrant, se importa `RagPipeline`.

In [7]:
pipeline = RagPipeline(
    collection="mmia6013_rag_baseline"
)

print("Backend:", os.environ["EMBEDDING_BACKEND"])
print("Modelo:", pipeline.encoder.modelo if hasattr(pipeline.encoder, "modelo") else "bge-m3")
print("Tope del modelo:", pipeline.max_seq_tokens)
print("Colección:", pipeline.collection)

/Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Backend: h200
Modelo: bge-m3
Tope del modelo: 8192
Colección: mmia6013_rag_baseline


#### 1. Ingesta. Parsea los documentos (PDF → texto). Documenta qué se pierde o se degrada (tablas, encabezados,pies de página) y qué documentos, si alguno, la ingesta rechazó por no tener texto útil.

Se usa la **Opción B del enunciado: papers del curso, en inglés** (un solo idioma declarado). Son seis PDF con capa de texto, 262 páginas en total, guardados en `corpus/`:

| Documento | Páginas |
|---|---:|
| Bai et al. (2022) — Constitutional AI | 34 |
| Brown et al. (2020) — GPT-3 | 75 |
| Ouyang et al. (2022) — InstructGPT | 68 |
| Rafailov et al. (2023) — DPO | 27 |
| Vaswani et al. (2017) — Attention Is All You Need | 15 |
| Wei et al. (2022) — Chain-of-Thought | 43 |

La ingesta transforma cada PDF en texto y, como en la Parte 0.a, rechaza (con aviso) los documentos sin texto útil.

In [8]:
corpus_dir = ROOT / "corpus"

print("Corpus:", corpus_dir)
print("¿Existe?:", corpus_dir.exists())
print("¿Es carpeta?:", corpus_dir.is_dir())

documentos = sorted(corpus_dir.glob("*"))

print("\nDocumentos encontrados:")
for doc in documentos:
    print(" -", doc.name)

Corpus: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/corpus
¿Existe?: True
¿Es carpeta?: True

Documentos encontrados:
 - .gitkeep
 - bai-2022-constitutional-ai.pdf
 - brown-2020-gpt3.pdf
 - ouyang-2022-instructgpt.pdf
 - rafailov-2023-dpo.pdf
 - vaswani-2017-attention-is-all-you-need.pdf
 - wei-2022-chain-of-thought.pdf


In [9]:
chunks = pipeline.ingest(
    corpus_dir,
    chunk_tokens=512,
    overlap_tokens=102
)

print(f"\nFragmentos obtenidos: {len(chunks)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


ingesta: bai-2022-constitutional-ai.pdf: 34 página(s), 94294 caracteres útiles → 75 fragmento(s)
ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: ouyang-2022-instructgpt.pdf: 68 página(s), 143167 caracteres útiles → 118 fragmento(s)
ingesta: rafailov-2023-dpo.pdf: 27 página(s), 71700 caracteres útiles → 66 fragmento(s)
ingesta: vaswani-2017-attention-is-all-you-need.pdf: 15 página(s), 31566 caracteres útiles → 25 fragmento(s)
ingesta: wei-2022-chain-of-thought.pdf: 43 página(s), 105503 caracteres útiles → 90 fragmento(s)

Fragmentos obtenidos: 530


Se generaron 530 fragmentos de 512 tokens con 102 tokens de solapamiento. **Ningún documento fue rechazado**: los seis tienen capa de texto (no aparece ningún `AVISO ingesta … NO se indexa`).

La advertencia `Token indices sequence length is longer than the specified maximum sequence length (30809 > 8192)` **no es un truncamiento**: la emite el tokenizador cuando la ingesta cuenta los tokens del documento entero para poder cortarlo. Al modelo solo llegan fragmentos de 512 tokens.

In [10]:
datos_chunks = []

for chunk in chunks:
    datos_chunks.append({
        "chunk_id": chunk.id,
        "documento": chunk.document,
        "chunk_index": chunk.chunk_index,
        "tokens": len(
            pipeline.encoder.tokenizer(
                chunk.text,
                add_special_tokens=False,
                truncation=False
            )["input_ids"]
        ),
        "texto": chunk.text
    })

df_chunks = pd.DataFrame(datos_chunks)
print("Total chunks:", len(df_chunks))

Total chunks: 530


Se construye una tabla para revisar los fragmentos generados durante la ingesta.

Para cada fragmento se registra:

- identificador;
- documento de origen;
- posición dentro del documento;
- número de tokens;
- texto extraído.

In [11]:
df_chunks.head()

,chunk_id,documento,chunk_index,tokens,texto
0,bai-2022-constitutional-ai-0000,bai-2022-constitutional-ai.pdf,0,512,[page=1] Constitutional AI: Harmlessness from ...
1,bai-2022-constitutional-ai-0001,bai-2022-constitutional-ai.pdf,1,512,"from the finetuned model, use a model to evalu..."
2,bai-2022-constitutional-ai-0002,bai-2022-constitutional-ai.pdf,2,510,"(RL) stage, shown as the sequence of steps at ..."
3,bai-2022-constitutional-ai-0003,bai-2022-constitutional-ai.pdf,3,512,"ying a general AI system, we cannot avoid choo..."
4,bai-2022-constitutional-ai-0004,bai-2022-constitutional-ai.pdf,4,512,+ Harmless Figure 2 We show harmlessness versu...


In [12]:
carpeta_resultados = ROOT / "resultados" / "parte1"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo_chunks = carpeta_resultados / "numeral1.csv"

df_chunks.to_csv(
    archivo_chunks,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_chunks}")

Guardado en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte1/numeral1.csv


#### 1.1 — Qué se pierde o se degrada en la ingesta

`pypdf` extrae el texto en el orden en que está dibujado en la página. Se mide sobre los 530 fragmentos qué estructura se perdió:

In [13]:
texto_total = " ".join(df_chunks["texto"])

degradaciones = {
    "marcas [page=N] (las añade la ingesta)": r"\[page=\d+\]",
    "palabras cortadas por guion de fin de línea (ej. 'Chris- tiano')": r"\b[a-z]+- [a-z]+\b",
    "pies de página / cabeceras de arXiv": r"arXiv:\d",
    "correos de autores": r"\S+@\S+\.\w+",
    "menciones a tablas (su contenido queda aplanado)": r"\bTable \d+",
    "menciones a figuras (la figura no se extrae)": r"\bFigure \d+",
}
df_degradacion = pd.DataFrame(
    [{"degradación": k, "ocurrencias": len(re.findall(p, texto_total))} for k, p in degradaciones.items()]
)
display(df_degradacion)

# El primer fragmento de cada paper: cuánto texto de cabecera (título, autores,
# afiliaciones, correos) hay antes de "Abstract".
primeros = df_chunks[df_chunks["chunk_index"] == 0]
cabecera = pd.DataFrame([{
    "documento": r["documento"],
    "caracteres_antes_de_Abstract": r["texto"].find("Abstract"),
    "caracteres_del_fragmento": len(r["texto"]),
    "inicio": r["texto"][:120] + "…",
} for _, r in primeros.iterrows()])
display(cabecera)

# Ejemplo de tabla aplanada: una fila de números sin encabezados de columna
for _, r in df_chunks.iterrows():
    m = re.search(r"(?:\d+\.\d+\s+){8,}", r["texto"])
    if m:
        print("Tabla aplanada en", r["chunk_id"], ":")
        print("   ", r["texto"][max(0, m.start() - 120):m.end()][:300])
        break

df_degradacion.to_csv(carpeta_resultados / "numeral1_degradacion.csv", index=False, encoding="utf-8-sig")

,degradación,ocurrencias
0,marcas [page=N] (las añade la ingesta),322
1,palabras cortadas por guion de fin de línea (e...,276
2,pies de página / cabeceras de arXiv,244
3,correos de autores,12
4,menciones a tablas (su contenido queda aplanado),199
5,menciones a figuras (la figura no se extrae),295


,documento,caracteres_antes_de_Abstract,caracteres_del_fragmento,inicio
0,bai-2022-constitutional-ai.pdf,844,1887,[page=1] Constitutional AI: Harmlessness from ...
1,brown-2020-gpt3.pdf,507,2098,[page=1] Language Models are Few-Shot Learners...
2,ouyang-2022-instructgpt.pdf,361,2144,[page=1] Training language models to follow in...
3,rafailov-2023-dpo.pdf,273,2131,[page=1] Direct Preference Optimization: Your ...
4,vaswani-2017-attention-is-all-you-need.pdf,599,2153,[page=1] Provided proper attribution is provid...
5,wei-2022-chain-of-thought.pdf,244,1909,[page=1] Chain-of-Thought Prompting Elicits Re...


Tabla aplanada en brown-2020-gpt3-0046 :
    erformance of the full GPT-3 significantly 22 [page=23] Setting 2D+ 2D- 3D+ 3D- 4D+ 4D- 5D+ 5D- 2Dx 1DC GPT-3 Zero-shot 76.9 58.0 34.2 48.3 4.0 7.5 0.7 0.8 19.8 9.8 


**Qué se pierde, con la evidencia de arriba:**

- **Tablas:** se aplanan en una secuencia de números sin encabezados (ver el ejemplo de GPT-3). Una pregunta sobre una cifra de una tabla puede recuperar el fragmento, pero sin saber a qué fila y columna corresponde cada número.
- **Figuras:** no se extraen; solo queda el pie de figura ("Figure N …").
- **Ecuaciones:** quedan como texto roto (subíndices y símbolos sueltos), notable en DPO.
- **Palabras cortadas por guion de fin de línea** ("Chris- tiano"): el tokenizador las ve como dos piezas.
- **Encabezados y pies de página** (arXiv, número de página) se mezclan con el cuerpo del texto.
- **Cabecera del primer fragmento:** título, lista de autores, afiliaciones y correos ocupan cientos de caracteres antes del `Abstract`. Es relevante para la Parte 2: el resumen, donde el golden set anota varias frases esperadas, comparte vector con esa cabecera.
- **Documentos rechazados:** ninguno.

### 2. Fragmentación. Baseline de tamaño fijo con solapamiento, en tokens del modelo de embeddings, con `chunk_tokens` y  `overlap_tokens` explícitos. El tope de bge-m3 es 8192 y el  Lab-02 del día miércoles corta por defecto en 512: la justificación dice por qué tu tamaño sirve a tus preguntas, no solo que cabe.

El baseline utiliza fragmentación de tamaño fijo con solapamiento. Se establece que la fragmentación debe expresarse en tokens del modelo de embeddings. En este caso, `bge-m3` tiene un límite de 8192 tokens.

Se muestran  los parámetros de fragmentación utilizados.

In [14]:
CHUNK_TOKENS = 512
OVERLAP_TOKENS = 102

print("Parámetros de fragmentación")
print("chunk_tokens:", CHUNK_TOKENS)
print("overlap_tokens:", OVERLAP_TOKENS)
print("máximo bge-m3:", pipeline.max_seq_tokens)

Parámetros de fragmentación
chunk_tokens: 512
overlap_tokens: 102
máximo bge-m3: 8192


**Justificación de `chunk_tokens=512` y `overlap_tokens=102`:**

- **Por qué 512 sirve a nuestras preguntas y no solo "cabe".** Las preguntas de prueba y del golden set piden afirmaciones concretas de un paper: qué arquitectura propone, qué objetivo de entrenamiento usa, en qué etapas consiste un método. En estos papers un párrafo ocupa del orden de 100–250 tokens y el resumen unos 250–300; con 512 tokens cada fragmento lleva la afirmación con su contexto inmediato (2–3 párrafos), y su vector sigue representando **una** idea.
- **Por qué no más grande.** El tope de bge-m3 (8192) equivale a casi todo el paper de Vaswani (25 fragmentos de 512). Un fragmento así promediaría introducción, método y resultados en un solo vector: cualquier pregunta lo encontraría "medio parecido", y al generador le llegarían miles de tokens irrelevantes por cada fragmento recuperado (con k=5, más de lo que conviene poner en el prompt).
- **Por qué no más chico.** Con fragmentos de ~128 tokens, las preguntas multi-fragmento (que combinan dos papers) necesitarían muchos más vecinos para reunir el contexto, y las definiciones que ocupan un párrafo entero quedarían partidas.
- **Solapamiento de un quinto (102 tokens).** Una afirmación que cae en el borde entre dos fragmentos aparece completa en al menos uno de ellos.

Es el punto de partida (baseline), no un óptimo: la Parte 2 mide si este tamaño recupera lo que el golden set espera.

###  3. Embedding e indexación. Con bge-m3 en la H200 (EMBEDDING_BACKEND=h200, el valor por defecto, con GlobalProtect conectada) o, sin VPN, con OpenAI (EMBEDDING_BACKEND=openai). Declara en el informe la fila por su id y con su verified_at. Indexa en Qdrant (el contenedor, no :memory:, que es solo para depurar).

En esta etapa se generan los vectores de los fragmentos utilizando `BAAI/bge-m3` en la H200 de la universidad

El resultado se almacena en Qdrant mediante la colección definida para este baseline.

Se utiliza el método `pipeline.index()` proporcionado por `rag_pipeline.py`.

Este método genera los embeddings de los fragmentos y los almacena en la colección de Qdrant.

In [15]:
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)
print("Número de chunks indexados:", len(chunks))

Índice creado correctamente.
Colección: mmia6013_rag_baseline
Número de chunks indexados: 530


Se consulta la colección creada en Qdrant para verificar que la indexación se haya realizado correctamente.

In [16]:
info = cliente_qdrant.get_collection(
    collection_name=pipeline.collection
)

print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=530 segments_count=6 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prev

Se obtiene el número  de vectores almacenados en la colección y se compara con el número de fragmentos generados durante la ingesta, podemos ver que es el mismo.

In [17]:
cantidad = cliente_qdrant.count(
    collection_name=pipeline.collection,
    exact=True
)

print("Vectores en Qdrant:", cantidad.count)
print("Fragmentos generados:", len(chunks))

Vectores en Qdrant: 530
Fragmentos generados: 530


A partir de las variables definidas previamente se construye un DataFrame con la configuración del embedding utilizado.

In [18]:
# Fila de la tabla semestral de modelos, clave "embeddings", tal como la cita el anexo del
# enunciado del Taller 02 (generado desde la tabla oficial el 2026-09-23).
# verified_at es la fecha de VERIFICACIÓN DE LA FILA, no la fecha en que corrimos el notebook.
# Nota: fuentes/modelos/modelos-2026-1.json de este repo es el archivo de EJEMPLO del curso
# (no trae la clave "embeddings"); por eso la fila se toma del anexo.
df_embedding = pd.DataFrame([{
    "id": "embed_local_multilingue",
    "proveedor": "BAAI",
    "modelo": "BAAI/bge-m3",
    "dimension": 1024,
    "tokens_secuencia": 8192,
    "multilingue": True,
    "costo_usd_1M": 0,
    "verified_at": "2026-08-27",
    "fuente": "Anexo del enunciado Taller 02 (tabla semestral, clave embeddings)",
}])

In [19]:
df_embedding

,id,proveedor,modelo,dimension,tokens_secuencia,multilingue,costo_usd_1M,verified_at,fuente
0,embed_local_multilingue,BAAI,BAAI/bge-m3,1024,8192,True,0,2026-08-27,Anexo del enunciado Taller 02 (tabla semestral...


In [20]:
archivo_embedding = carpeta_resultados / "numeral3_embedding_baseline.csv"

df_embedding.to_csv(
    archivo_embedding,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_embedding}")

Guardado en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte1/numeral3_embedding_baseline.csv


### 4. Recuperación. Muestra los top-5 fragmentos para al menos 3 preguntas de prueba, con documento y puntaje.

El cuarto paso consiste en consultar el índice vectorial y recuperar los cinco fragmentos más similares para cada pregunta de prueba.

Se incluyen tres preguntas en inglés y relacionadas con los papers principales del corpus.
Las preguntas buscan información conceptual específica y permiten comprobar si el índice recupera los documentos correspondientes.

In [21]:
preguntas_prueba = [
    "What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?",

    "How does GPT-3 demonstrate that increasing model size and using in-context learning can improve few-shot performance?",

    "How does InstructGPT use human feedback to improve the alignment of language models with user intentions?"
]

In [22]:
print("Número de preguntas:", len(preguntas_prueba))

Número de preguntas: 3


Para cada pregunta se ejecuta `pipeline.retrieve()` con `top_k=5`.

Para cada vecino recuperado se muestran:

- posición;
- puntaje de similitud;
- documento;
- `chunk_id`;
- texto recuperado.

Estos resultados permiten analizar el comportamiento del retrieval antes de incorporar la generación.

In [23]:
resultados_retrieval = []

for pregunta in preguntas_prueba:

    resultados = pipeline.retrieve(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)
    print("\nTOP 5 VECINOS:\n")

    for rank, resultado in enumerate(resultados, start=1):

        print(f"--- Vecino {rank} ---")
        print(f"Score:     {resultado['score']:.4f}")
        print(f"Documento: {resultado['document']}")
        print(f"Chunk ID:  {resultado['chunk_id']}")
        print(f"Texto:    {resultado['text']}")
        print()

        resultados_retrieval.append({
            "pregunta": pregunta,
            "rank": rank,
            "score": resultado["score"],
            "documento": resultado["document"],
            "chunk_id": resultado["chunk_id"],
            "texto": resultado["text"]
        })

PREGUNTA:
What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?

TOP 5 VECINOS:

--- Vecino 1 ---
Score:     0.6861
Documento: vaswani-2017-attention-is-all-you-need.pdf
Chunk ID:  vaswani-2017-attention-is-all-you-need-0003
Texto:    it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent 

In [24]:
df_retrieval = pd.DataFrame(resultados_retrieval)

In [25]:
df_retrieval.head()

,pregunta,rank,score,documento,chunk_id,texto
0,What are the main advantages of the Transforme...,1,0.686089,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0003,it more difficult to learn dependencies betwee...
1,What are the main advantages of the Transforme...,2,0.670047,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0002,"they generate a sequence of hidden statesht, a..."
2,What are the main advantages of the Transforme...,3,0.664278,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0017,". During inference, we 9 [page=10] Table 4: Th..."
3,What are the main advantages of the Transforme...,4,0.636401,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0018,recurrent layers most commonly used in encoder...
4,What are the main advantages of the Transforme...,5,0.635783,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0000,[page=1] Provided proper attribution is provid...


Los resultados de las tres preguntas se consolidan en un DataFrame para facilitar su revisión y posterior análisis.

In [26]:
archivo_retrieval = carpeta_resultados / "numeral4_retrieval_top5.csv"

df_retrieval.to_csv(
    archivo_retrieval,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_retrieval}")

Guardado en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte1/numeral4_retrieval_top5.csv


### 5. Generación.`rag_pipeline.generate` ya trae la llamada: con la clave del curso usa la fila propietario_economico; sin clave, Ollama con open_weight_pequeno. El prompt del baseline hace tres cosas—delimita el contexto, obliga a usar solo el contexto y autoriza la abstención con una frase fija— y esa frase es la misma en el prompt, en el golden set y en el detector: ABSTENCION en cambiarla en los tres sitios

Se utiliza `generate()` de `rag_pipeline.py`. El enunciado ofrece dos rutas:

- con la clave del curso, la fila `propietario_economico` (OpenAI);
- sin clave, la fila `open_weight_pequeno` por Ollama.

**El grupo usa la segunda ruta en todo el taller**, con el Ollama de la H200 (`qwen3:32b`), a costo cero. Es el mismo generador de las Partes 2 y 3, para que las tasas de abstención sean comparables entre partes. El prompt del baseline delimita el contexto, obliga a usar solo el contexto y autoriza la abstención con la frase fija `ABSTENCION`, la misma del golden set y del detector.

In [27]:
# Verificación del generador ANTES de generar: si Ollama no responde, rag_pipeline cae en
# "modo inspección" sin lanzar ningún error, y la Parte 1.5 quedaría sin respuestas reales.
import urllib.request

with urllib.request.urlopen(f"{rag_pipeline.OLLAMA_URL}/api/tags", timeout=15) as r:
    modelos_servidos = [m["name"] for m in json.loads(r.read())["models"]]

modelo_generador = rag_pipeline.fila_de_la_tabla("models", "open_weight_pequeno").get("model")
print("Ollama:", rag_pipeline.OLLAMA_URL)
print("Modelo de la fila open_weight_pequeno:", modelo_generador)
assert modelo_generador in modelos_servidos, f"{modelo_generador} no está servido en la H200"
print("OK: el generador está disponible.")

Ollama: http://172.28.230.10:11434
Modelo de la fila open_weight_pequeno: qwen3:32b
OK: el generador está disponible.


Para cada pregunta se registra el generador utilizado, la respuesta obtenida y si el detector identificó una abstención.

In [28]:
resultados_generacion = []
salidas_generacion = []   # salida completa (con contextos y prompt) para el JSON

for pregunta in preguntas_prueba:
    resultado = pipeline.answer(pregunta, top_k=5)
    salidas_generacion.append(resultado)

    print("=" * 100)
    print("PREGUNTA:", pregunta)
    print("GENERADOR:", resultado["generator"])
    print("ABSTENCIÓN:", resultado["abstained"])
    print("\nRESPUESTA:")
    print(resultado["answer"])
    print()

    resultados_generacion.append({
        "pregunta": resultado["question"],
        "respuesta": resultado["answer"],
        "abstencion": resultado["abstained"],
        "generador": resultado["generator"],
        "prompt": resultado["prompt"],
    })

generadores = {r["generador"] for r in resultados_generacion}
assert all(g.startswith("ollama:") for g in generadores), f"Generador inesperado: {generadores}"

PREGUNTA: What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?
GENERADOR: ollama:qwen3:32b
ABSTENCIÓN: False

RESPUESTA:
El Transformer ofrece varias ventajas clave sobre modelos basados en redes recurrentes (RNN) o convolucionales (CNN) para transducción de secuencias:  
1. **Mayor paralelización**: Al eliminar la dependencia secuencial de RNNs/CNNs, el Transformer permite un entrenamiento significativamente más rápido (sección 4 y abstracto).  
2. **Dependencias globales sin restricciones de distancia**: A diferencia de CNNs (donde las operaciones crecen linealmente o logarítmicamente con la distancia entre posiciones), el Transformer relaciona posiciones arbitrarias en una secuencia con un número constante de operaciones, facilitando el aprendizaje de dependencias distantes (contexto 1 y 2).  
3. **Superioridad en rendimiento**: Alcanza resultados de vanguardia en tareas como traducción (28.

Las respuestas generadas se almacenan en un dataframe para conservar una salida reproducible de esta etapa.

In [29]:
df_generacion = pd.DataFrame(resultados_generacion)

display(df_generacion[
    [
        "pregunta",
        "respuesta",
        "abstencion",
        "generador"
    ]
])

,pregunta,respuesta,abstencion,generador
0,What are the main advantages of the Transforme...,El Transformer ofrece varias ventajas clave so...,False,ollama:qwen3:32b
1,How does GPT-3 demonstrate that increasing mod...,GPT-3 demuestra que aumentar el tamaño del mod...,False,ollama:qwen3:32b
2,How does InstructGPT use human feedback to imp...,InstructGPT utiliza retroalimentación humana e...,False,ollama:qwen3:32b


In [30]:
archivo_generacion = carpeta_resultados / "numeral5_generacion.csv"

df_generacion.to_csv(
    archivo_generacion,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_generacion}")

Guardado en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte1/numeral5_generacion.csv


In [31]:
# Se guarda la MISMA generación de la celda anterior (antes se volvía a generar aquí y el
# JSON podía no coincidir con el CSV).
archivo_generacion_json = carpeta_resultados / "numeral5_generacion.json"

with open(archivo_generacion_json, "w", encoding="utf-8") as f:
    json.dump(salidas_generacion, f, ensure_ascii=False, indent=2)

print(f"Guardado en: {archivo_generacion_json}")

Guardado en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte1/numeral5_generacion.json


## Conclusión

Se implementó el baseline RAG completo sobre seis papers del curso en inglés (262 páginas, 530 fragmentos):

**Ingesta → Fragmentación (512/102 tokens de bge-m3) → Embeddings (`embed_local_multilingue`, bge-m3, verificada 2026-08-27) → Qdrant (contenedor) → Recuperación top-5 → Generación (`open_weight_pequeno`, qwen3:32b en la H200)**

Lo que la ingesta degrada queda documentado en el numeral 1: tablas aplanadas, ecuaciones rotas, palabras cortadas por guion, cabeceras con autores y afiliaciones en el primer fragmento de cada paper, y pies de página de arXiv. La Parte 2 mide con el golden set si esto afecta a la recuperación.

**Pendiente de declarar:** la fila `open_weight_pequeno → qwen3:32b` de `fuentes/modelos/modelos-2026-1.json` la agregó el grupo (`verified_at: null`), porque ese archivo es el de ejemplo del curso. Se debe reemplazar por la fila de la tabla oficial.